# 02G — Protein domains


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📊 domain counts


In [3]:
dom_counts = d['domain_clean'].value_counts().head(30).reset_index()
dom_counts.columns = ['domain', 'count']
display(dom_counts)
fig = px.bar(dom_counts, x='domain', y='count', title='Domain counts')
fig.update_xaxes(tickangle=-45)
fig.show()


,domain,count
0,Interaction with SYNM,243
1,Spectrin 4,148
2,Spectrin 2,146
3,Spectrin 5,144
4,Calponin-homology (CH2) 2,140
5,Spectrin 1,132
6,Spectrin 6,128
7,Spectrin 3,126
8,Spectrin 12,120
9,Spectrin 15,119


## 2. 📊 domain pathogenic stacked 🛡


In [4]:
top = d['domain_clean'].value_counts().head(20).index
tmp = d[d['domain_clean'].isin(top)][['domain_clean', 'pathogenic']]
fig = px.histogram(tmp, x='domain_clean', color='pathogenic', barmode='stack', title='Domain x pathogenic')
fig.update_xaxes(tickangle=-45)
fig.show()


## 3. 📊 domain pathogenic fraction 🛡


In [5]:
tbl = d.groupby('domain_clean').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('var_id', 'size')).reset_index().sort_values('n', ascending=False).head(25)
display(tbl)
fig = px.bar(tbl, x='domain_clean', y='pathogenic_fraction', hover_data=['n'], title='Domain pathogenic fraction')
fig.update_xaxes(tickangle=-45)
fig.show()


,domain_clean,pathogenic_fraction,n
6,Interaction with SYNM,0.415638,243
25,Spectrin 4,0.351351,148
18,Spectrin 2,0.308219,146
26,Spectrin 5,0.291667,144
4,Calponin-homology (CH2) 2,0.314286,140
7,Spectrin 1,0.287879,132
27,Spectrin 6,0.273438,128
24,Spectrin 3,0.301587,126
10,Spectrin 12,0.283333,120
13,Spectrin 15,0.302521,119


## 4. 🧪 Fisher domain vs rest ⭐


In [6]:
top = d['domain_clean'].value_counts()[lambda s: s >= 20].index
rows = []
for dom in top:
    tmp = d[['domain_clean', 'pathogenic']].dropna().copy()
    tmp['is_domain'] = tmp['domain_clean'].eq(dom)
    tab, odds, p, ci = fisher_bool(tmp, 'is_domain', 'pathogenic')
    rows.append({'domain': dom, 'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]})
res = pd.DataFrame(rows).sort_values('p_value')
display(res.head(40))


,domain,odds_ratio,p_value,or_ci_low,or_ci_high
26,Disordered,0.257897,0.000122,0.117300,0.567015
0,Interaction with SYNM,1.611527,0.000546,1.234122,2.104345
30,Binds to SNTB1,3.498062,0.000673,1.691468,7.234211
13,Spectrin 9,0.528477,0.006665,0.329390,0.847897
29,Actin-binding,2.320477,0.013803,1.232913,4.367394
24,Spectrin 17,1.567942,0.061956,0.990311,2.482494
16,Spectrin 11,1.460036,0.067127,0.976946,2.182008
15,Spectrin 13,0.691660,0.112730,0.441804,1.082818
20,Spectrin 24,0.674658,0.152563,0.406048,1.120960
21,Spectrin 21,0.697075,0.185491,0.418743,1.160410


## 5. 📋 domain OR + q-values ⭐


In [7]:
top = d['domain_clean'].value_counts()[lambda s: s >= 20].index
rows = []
for dom in top:
    tmp = d[['domain_clean', 'pathogenic']].dropna().copy()
    tmp['is_domain'] = tmp['domain_clean'].eq(dom)
    tab, odds, p, ci = fisher_bool(tmp, 'is_domain', 'pathogenic')
    rows.append({'domain': dom, 'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]})
res = pd.DataFrame(rows)
res['q_value'] = bh_adjust(res['p_value'].values)
res = res.sort_values('q_value')
display(res.head(40))


,domain,odds_ratio,p_value,or_ci_low,or_ci_high,q_value
26,Disordered,0.257897,0.000122,0.117300,0.567015,0.003789
0,Interaction with SYNM,1.611527,0.000546,1.234122,2.104345,0.006952
30,Binds to SNTB1,3.498062,0.000673,1.691468,7.234211,0.006952
13,Spectrin 9,0.528477,0.006665,0.329390,0.847897,0.051651
29,Actin-binding,2.320477,0.013803,1.232913,4.367394,0.085578
16,Spectrin 11,1.460036,0.067127,0.976946,2.182008,0.297276
24,Spectrin 17,1.567942,0.061956,0.990311,2.482494,0.297276
15,Spectrin 13,0.691660,0.112730,0.441804,1.082818,0.436828
20,Spectrin 24,0.674658,0.152563,0.406048,1.120960,0.525494
21,Spectrin 21,0.697075,0.185491,0.418743,1.160410,0.575022


## 6. 📊 domain vs phenotype


In [8]:
top = d['domain_clean'].value_counts().head(15).index
tmp = d[d['domain_clean'].isin(top)]
heat = pd.crosstab(tmp['domain_clean'], tmp['phenotype'])
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Blues', title='Domain vs phenotype')
fig.show()


## 7. 🧪 χ² domain ~ phenotype


In [9]:
tmp = d[d['domain_clean'].isin(d['domain_clean'].value_counts()[lambda s: s >= 20].index)]
tab, chi2, p, dof = chi2_table(tmp, 'domain_clean', 'phenotype')
display(tab.head(30))
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


phenotype,BMD,DMD
domain_clean,,
Actin-binding,3,26
Binds to SNTB1,3,23
Calponin-homology (CH1) 1,10,87
Calponin-homology (CH2) 2,14,105
Disordered,5,50
Interaction with SYNM,26,178
Spectrin 1,7,104
Spectrin 10,2,40
Spectrin 11,8,82


,value
chi2,24.643724
dof,30.000000
p_value,0.742154


## 8. 📊 domain vs mutation_type


In [10]:
top = d['domain_clean'].value_counts().head(15).index
tmp = d[d['domain_clean'].isin(top)]
heat = pd.crosstab(tmp['domain_clean'], tmp['mutation_type'])
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Viridis', title='Domain vs mutation_type')
fig.show()


## 9. 🧪 χ² domain ~ mutation


In [11]:
tmp = d[d['domain_clean'].isin(d['domain_clean'].value_counts()[lambda s: s >= 20].index)]
tab, chi2, p, dof = chi2_table(tmp, 'domain_clean', 'mutation_type')
display(tab.head(30))
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


mutation_type,frameshift,large_deletion,missense,nonsense,splice
domain_clean,,,,,
Actin-binding,15,0,17,4,3
Binds to SNTB1,12,0,11,7,1
Calponin-homology (CH1) 1,13,0,70,18,12
Calponin-homology (CH2) 2,14,0,86,31,7
Disordered,3,0,52,6,2
Interaction with SYNM,47,0,125,58,9
Spectrin 1,10,0,80,32,6
Spectrin 10,1,0,33,12,1
Spectrin 11,16,0,52,29,6


,value
chi2,2.233882e+02
dof,1.200000e+02
p_value,3.172493e-08


## 10. 📊 rod vs binding domain ⭐


In [12]:
tmp = d[['domain_clean', 'pathogenic']].copy()
tmp['domain_class'] = np.where(tmp['domain_clean'].str.contains('spectrin', case=False, na=False), 'rod', np.where(tmp['domain_clean'].str.contains('interaction|actin|binding|ch1|ch2|ww|cysteine', case=False, regex=True, na=False), 'binding', 'other'))
cls = tmp[tmp['domain_class'].isin(['rod', 'binding'])]
tbl = cls.groupby('domain_class').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('pathogenic', 'size')).reset_index()
display(tbl)
fig = px.bar(tbl, x='domain_class', y='pathogenic_fraction', hover_data=['n'], title='Rod vs binding domain pathogenic fraction')
fig.show()


,domain_class,pathogenic_fraction,n
0,binding,0.366171,538
1,rod,0.306139,2525


## 11. 🧪 Fisher rod vs binding ⭐


In [13]:
tmp = d[['domain_clean', 'pathogenic']].copy()
tmp['domain_class'] = np.where(tmp['domain_clean'].str.contains('spectrin', case=False, na=False), 'rod', np.where(tmp['domain_clean'].str.contains('interaction|actin|binding|ch1|ch2|ww|cysteine', case=False, regex=True, na=False), 'binding', 'other'))
cls = tmp[tmp['domain_class'].isin(['rod', 'binding'])].copy()
cls['is_rod'] = cls['domain_class'].eq('rod')
tab, odds, p, ci = fisher_bool(cls, 'is_rod', 'pathogenic')
display(tab)
display(pd.Series({'odds_ratio_rod_vs_binding': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


pathogenic,False,True
is_rod,,
False,341,197
True,1752,773


,value
odds_ratio_rod_vs_binding,0.763719
p_value,0.007901
or_ci_low,0.628570
or_ci_high,0.927926


## 12. 📊 domain treemap


In [14]:
tbl = d.groupby('domain_clean').agg(n=('var_id', 'size'), pathogenic_fraction=('pathogenic', 'mean')).reset_index().sort_values('n', ascending=False).head(40)
fig = px.treemap(tbl, path=['domain_clean'], values='n', color='pathogenic_fraction', title='Domain treemap')
fig.show()


## 13. 📊 domain size vs pathogenic fraction


In [15]:
tmp = d.dropna(subset=['domain_clean', 'aa_pos_num']).copy()
span = tmp.groupby('domain_clean').agg(start=('aa_pos_num', 'min'), end=('aa_pos_num', 'max'), pathogenic_fraction=('pathogenic', 'mean'), n=('var_id', 'size')).reset_index()
span['domain_size_est'] = span['end'] - span['start'] + 1
span = span[span['n'] >= 10]
display(span.sort_values('n', ascending=False).head(30))
fig = px.scatter(span, x='domain_size_est', y='pathogenic_fraction', size='n', text='domain_clean', title='Domain size vs pathogenic fraction')
fig.show()


,domain_clean,start,end,pathogenic_fraction,n,domain_size_est
6,Interaction with SYNM,1415.0,3408.0,0.415638,243,1994.0
25,Spectrin 4,720.0,828.0,0.351351,148,109.0
18,Spectrin 2,448.0,555.0,0.308219,146,108.0
26,Spectrin 5,830.0,934.0,0.291667,144,105.0
4,Calponin-homology (CH2) 2,134.0,239.0,0.314286,140,106.0
7,Spectrin 1,340.0,447.0,0.287879,132,108.0
27,Spectrin 6,943.0,1045.0,0.273438,128,103.0
24,Spectrin 3,560.0,665.0,0.301587,126,106.0
10,Spectrin 12,1574.0,1676.0,0.283333,120,103.0
28,Spectrin 7,1048.0,1154.0,0.294118,119,107.0


## 14. 🧪 Spearman domain size vs pathogenic 📖


In [16]:
tmp = d.dropna(subset=['domain_clean', 'aa_pos_num']).copy()
span = tmp.groupby('domain_clean').agg(start=('aa_pos_num', 'min'), end=('aa_pos_num', 'max'), pathogenic_fraction=('pathogenic', 'mean'), n=('var_id', 'size')).reset_index()
span['domain_size_est'] = span['end'] - span['start'] + 1
span = span[span['n'] >= 10]
_, rho, p = spearman_xy(span, 'domain_size_est', 'pathogenic_fraction')
display(pd.Series({'n_domains': len(span), 'spearman_rho': rho, 'p_value': p}).to_frame('value'))


,value
n_domains,31.000000
spearman_rho,0.275752
p_value,0.133223


## 15. 📊 domain entropy 📖


In [17]:
p = d['domain_clean'].value_counts(normalize=True)
ent = entropy(p.values, base=2)
display(pd.Series({'n_domains': int(p.shape[0]), 'entropy_bits': float(ent)}).to_frame('value'))


,value
n_domains,33.000000
entropy_bits,4.852372


## 16. 📊 domain positional ordering


In [18]:
tmp = d.dropna(subset=['domain_clean', 'aa_pos_num']).copy()
order_tbl = tmp.groupby('domain_clean').agg(median_aa=('aa_pos_num', 'median'), n=('var_id', 'size')).reset_index().sort_values('median_aa')
fig = px.bar(order_tbl, x='domain_clean', y='median_aa', hover_data=['n'], title='Domain positional ordering by median aa')
fig.update_xaxes(tickangle=-45)
fig.show()
